# Fine-tune Qwen2.5-3B-Instruct with Unsloth (SFT) — Kaggle

Fine-tunes `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` using Supervised Fine-Tuning (SFT) with Unsloth.

**Setup:**
1. Go to **Settings → Accelerator → GPU (T4 × 2) or P100**
2. Wait for GPU to connect
3. Run all cells in order

**Kaggle GPU Limits:** 30 hours/week for T4. Plan your training accordingly.

**Based on:** [unsloth-buddy](https://github.com/TYH-labs/unsloth-buddy) skill

## Cell 1: Install Unsloth

In [ ]:
%%capture
!pip install unsloth
# If you get errors, try:
# !pip install unsloth --no-deps
# !pip install --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

## Cell 2: Verify GPU

In [ ]:
import torch, json
assert torch.cuda.is_available(), "No GPU! Go to Settings → Accelerator → GPU"

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

from unsloth import FastLanguageModel
import unsloth, trl, transformers, datasets

print(json.dumps({
    "gpu": gpu_name,
    "vram_gb": round(vram_gb, 1),
    "unsloth": unsloth.__version__,
    "trl": trl.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "cuda": torch.version.cuda,
))
print("GPU ready!")

## Cell 3: Load Model & Apply LoRA

In [ ]:
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print(f"Model loaded. Trainable params: {model.print_trainable_parameters()}")

## Cell 4: Prepare Dataset

Choose ONE option below.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# OPTION A: Sample HuggingFace dataset (quick demo)
# ══════════════════════════════════════════════════════════════════════════════

from datasets import load_dataset

dataset = load_dataset("BAAI/Infinity-Instruct", split="train", streaming=True)
dataset = dataset.shuffle(seed=42).select(range(500))

def format_chat(example):
    instruction = example.get("instruction", example.get("query", ""))
    output = example.get("output", example.get("response", ""))
    if not instruction or not output:
        return None
    messages = [
        {"role": "user", "content": instruction},
        {"role": "assistant", "content": output},
    ]
    return {"messages": messages}

dataset = dataset.map(format_chat, remove_columns=dataset.column_names)
dataset = dataset.filter(lambda x: x is not None and x.get("messages") is not None)

print(f"Dataset: {len(dataset)} samples")
print(f"Example: {dataset[0]}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# OPTION B: Upload your own dataset (uncomment and modify)
# ══════════════════════════════════════════════════════════════════════════════

# from kaggle_secrets import UserSecretsClient
# secrets = UserSecretsClient()
# # Or upload via the Kaggle data browser (Add Data → Upload)
#
# import pandas as pd
# df = pd.read_json("/kaggle/input/your-dataset/data.jsonl", lines=True)
#
# # Adapt column names to your data:
# def format_my_data(example):
#     return {"messages": [
#         {"role": "user", "content": example["question"]},
#         {"role": "assistant", "content": example["answer"]},
#     ]}
#
# from datasets import Dataset
# dataset = Dataset.from_pandas(df).map(format_my_data, remove_columns=df.columns.tolist())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# OPTION C: Create a custom dataset inline
# ══════════════════════════════════════════════════════════════════════════════

# from datasets import Dataset
#
# my_data = [
#     {"messages": [{"role": "user", "content": "What is Python?"},
#                    {"role": "assistant", "content": "Python is a high-level programming language..."}]},
#     # Add more examples (aim for 100+ for real training)
# ]
#
# dataset = Dataset.from_list(my_data)

## Cell 5: Train with SFT

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        max_steps = 200,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        warmup_steps = 10,
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print(f"Training complete! Final loss: {trainer_stats.metrics['train_loss']:.4f}")

## Cell 6: Save LoRA Adapters

In [ ]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("Adapters saved to lora_model/")

import os
for f in os.listdir("lora_model"):
    size = os.path.getsize(os.path.join("lora_model", f))
    print(f"  {f}: {size / 1e6:.1f} MB")

## Cell 7: Test Inference

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_model",
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "Explain what machine learning is in simple terms."},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, top_p=0.9)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

## Cell 8 (Optional): Export to GGUF

In [ ]:
# Uncomment to export:
# model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method="q4_k_m")
# print("GGUF exported!")
#
# Download:
# import glob
# from IPython.display import FileLink
# for f in glob.glob("model_gguf/*.gguf"):
#     print(f"Download: {f}")
#     display(FileLink(f))

## Download Adapters

Download `lora_model/` from the Kaggle Output panel (right side → Output tab → Download).